# NAVYA — Testing Model Output with App Inputs

This notebook tests the final exported app model using the same fields the mobile application provides. It does not train, tune, or overwrite any model. Run Notebook 10 first so `models/cycle_length_model.joblib` and its metadata represent the locked final model.

## 1. App input contract

The app supplies cycle and period histories **oldest to newest**. For three or more completed cycles, the newest three are mapped to the model fields in reverse order: `prev_cycle_1` is the most recent cycle, then `prev_cycle_2`, then `prev_cycle_3`.

In [6]:
from pathlib import Path
from datetime import date, datetime
import json
import joblib
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'models').exists():
    ROOT = ROOT.parent

MODEL_PATH = ROOT / 'models' / 'cycle_length_model.joblib'
METADATA_PATH = ROOT / 'models' / 'cycle_length_model_metadata.json'
if not MODEL_PATH.exists() or not METADATA_PATH.exists():
    raise FileNotFoundError('Run Notebook 10 first: final exported model or metadata is missing.')

pipeline = joblib.load(MODEL_PATH)
metadata = json.loads(METADATA_PATH.read_text())
FEATURES = metadata['feature_order']
actual_estimator = pipeline.named_steps['model'].__class__.__name__
if metadata['selected_estimator'] != actual_estimator:
    raise RuntimeError('Model and metadata do not match. Re-run Notebook 10 before testing.')

print('Loaded final estimator:', actual_estimator)
print('Target:', metadata['target'])
print('Feature count/order check:', len(FEATURES), 'features')
display(pd.DataFrame({'position': range(1, len(FEATURES) + 1), 'feature': FEATURES}))

Loaded final estimator: XGBRegressor
Target: next_cycle_length_days
Feature count/order check: 18 features


,position,feature
0,1,age_years
1,2,menarche_age
2,3,height_cm
3,4,weight_kg
4,5,bmi
5,6,sleep_hours
6,7,stress_level
7,8,exercise_frequency
8,9,uses_medication_or_contraceptive
9,10,prev_cycle_1


## 2. Validate app fields and build the exact feature row

This cell calculates age from date of birth, recalculates BMI, validates values, and builds the exact 18 numerical columns expected by the exported pipeline. For period-history gaps, the available mean is used; if no period history exists, the training median in the metadata is used. Cycle histories with fewer than three values do **not** use the ML model—they use the fallback policy later in this notebook.

In [7]:
def require_number(name, value, low, high, integer=False):
    if isinstance(value, bool) or not isinstance(value, (int, float, np.integer, np.floating)) or not np.isfinite(value):
        raise ValueError(f'{name} must be a finite number.')
    if integer and int(value) != value:
        raise ValueError(f'{name} must be an integer.')
    if not low <= value <= high:
        raise ValueError(f'{name} must be between {low} and {high}.')
    return int(value) if integer else float(value)

def validate_history(name, values, low, high):
    if not isinstance(values, list):
        raise ValueError(f'{name} must be a list ordered oldest-to-newest.')
    return [require_number(f'{name}[{i}]', value, low, high) for i, value in enumerate(values)]

def period_features(values):
    values = validate_history('period_lengths', values, 1, 14)[-3:]
    fill = float(np.mean(values)) if values else float(metadata['training_period_history_median'])
    padded_oldest_to_newest = values + [fill] * (3 - len(values))
    return list(reversed(padded_oldest_to_newest))

def build_feature_row(payload, today=None):
    required = {'date_of_birth', 'menarche_age', 'height_cm', 'weight_kg', 'sleep_hours', 'stress_level', 'exercise_frequency', 'uses_medication_or_contraceptive', 'cycle_lengths', 'period_lengths'}
    missing = required - set(payload)
    if missing:
        raise ValueError(f'Missing required fields: {sorted(missing)}')
    try:
        dob = datetime.strptime(payload['date_of_birth'], '%Y-%m-%d').date()
    except (TypeError, ValueError):
        raise ValueError('date_of_birth must use YYYY-MM-DD.')
    today = today or date.today()
    age = today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))
    age = require_number('age_years', age, 8, 100, integer=True)
    menarche = require_number('menarche_age', payload['menarche_age'], 7, age, integer=True)
    height = require_number('height_cm', payload['height_cm'], 100, 230)
    weight = require_number('weight_kg', payload['weight_kg'], 25, 250)
    sleep = require_number('sleep_hours', payload['sleep_hours'], 1, 24)
    stress = require_number('stress_level', payload['stress_level'], 1, 5, integer=True)
    exercise = require_number('exercise_frequency', payload['exercise_frequency'], 0, 2, integer=True)
    medication = payload['uses_medication_or_contraceptive']
    if not isinstance(medication, bool):
        raise ValueError('uses_medication_or_contraceptive must be true or false.')
    cycles = validate_history('cycle_lengths', payload['cycle_lengths'], 15, 60)
    if len(cycles) < 3:
        return None, cycles
    recent_cycles = cycles[-3:]
    prev_cycle_1, prev_cycle_2, prev_cycle_3 = reversed(recent_cycles)
    prev_period_1, prev_period_2, prev_period_3 = period_features(payload['period_lengths'])
    row = {
        'age_years': age, 'menarche_age': menarche, 'height_cm': height, 'weight_kg': weight,
        'bmi': weight / (height / 100) ** 2, 'sleep_hours': sleep, 'stress_level': stress,
        'exercise_frequency': exercise, 'uses_medication_or_contraceptive': int(medication),
        'prev_cycle_1': prev_cycle_1, 'prev_cycle_2': prev_cycle_2, 'prev_cycle_3': prev_cycle_3,
        'avg_previous_cycle_length': float(np.mean(recent_cycles)),
        'std_previous_cycle_length': float(np.std(recent_cycles, ddof=0)),
        'prev_period_1': prev_period_1, 'prev_period_2': prev_period_2, 'prev_period_3': prev_period_3,
        'avg_previous_period_length': float(np.mean([prev_period_1, prev_period_2, prev_period_3])),
    }
    features = pd.DataFrame([[row[name] for name in FEATURES]], columns=FEATURES)
    assert features.columns.tolist() == FEATURES
    return features, cycles

## 3. Predict safely from an app payload

For three or more supported cycles, the exported ML pipeline produces a point prediction and 90% conformal interval. Clearly unsupported histories use the user's recent-cycle average and are labelled as a fallback. Zero, one, or two completed cycles also use labelled fallbacks.

In [8]:
def predict_from_app_payload(payload):
    features, cycles = build_feature_row(payload)
    if len(cycles) == 0:
        return {'predicted_cycle_length_days': float(metadata['training_cycle_history_median']), 'rounded_prediction': round(float(metadata['training_cycle_history_median'])), 'prediction_method': 'fallback_population_median', 'prediction_status': 'not_enough_personal_data', 'feature_row': None}
    if len(cycles) < 3:
        estimate = float(np.mean(cycles))
        return {'predicted_cycle_length_days': estimate, 'rounded_prediction': round(estimate), 'prediction_method': 'fallback_history_average', 'prediction_status': 'not_enough_cycle_history', 'feature_row': None}
    recent_cycles = cycles[-3:]
    lower = float(metadata['supported_cycle_lower_bound'])
    upper = float(metadata['supported_cycle_upper_bound'])
    if min(recent_cycles) < lower or max(recent_cycles) > upper:
        estimate = float(np.mean(recent_cycles))
        return {'predicted_cycle_length_days': estimate, 'rounded_prediction': round(estimate), 'prediction_method': 'fallback_history_average', 'prediction_status': 'outside_training_distribution', 'supported_cycle_range_days': [lower, upper], 'feature_row': features}
    raw_prediction = float(pipeline.predict(features)[0])
    prediction = float(np.clip(raw_prediction, 15, 60))
    q_hat = float(metadata['q_hat_days'])
    return {'raw_prediction': raw_prediction, 'predicted_cycle_length_days': prediction, 'rounded_prediction': round(prediction), 'prediction_method': 'machine_learning', 'prediction_status': 'supported', 'prediction_interval': {'lower_days': max(15.0, prediction - q_hat), 'upper_days': min(60.0, prediction + q_hat), 'coverage': float(metadata['coverage'])}, 'feature_row': features}

def show_prediction(payload):
    result = predict_from_app_payload(payload)
    display(pd.DataFrame([{key: value for key, value in result.items() if key != 'feature_row'}]))
    if result['feature_row'] is not None:
        print('Exact feature row supplied to the model:')
        display(result['feature_row'])
    return result

## 4. Test the example app payload

This is the same JSON-shaped input provided by the app. Edit the values and run this cell to test another realistic user input.

In [9]:
app_payload = {
    'date_of_birth': '2000-08-15',
    'menarche_age': 13,
    'height_cm': 165.1,
    'weight_kg': 60.0,
    'sleep_hours': 6,
    'stress_level': 3,
    'exercise_frequency': 1,
    'uses_medication_or_contraceptive': False,
    'cycle_lengths': [28, 29, 27],
    'period_lengths': [5, 5, 6],
}
result = show_prediction(app_payload)

,raw_prediction,predicted_cycle_length_days,rounded_prediction,prediction_method,prediction_status,prediction_interval
0,27.718315,27.718315,28,machine_learning,supported,"{'lower_days': 25.545330047607422, 'upper_days..."


Exact feature row supplied to the model:


,age_years,menarche_age,height_cm,weight_kg,bmi,sleep_hours,stress_level,exercise_frequency,uses_medication_or_contraceptive,prev_cycle_1,prev_cycle_2,prev_cycle_3,avg_previous_cycle_length,std_previous_cycle_length,prev_period_1,prev_period_2,prev_period_3,avg_previous_period_length
0,26,13,165.1,60.0,22.011878,6.0,3,1,0,27.0,29.0,28.0,28.0,0.816497,6.0,5.0,5.0,5.333333


## 5. Interactive prediction tester — enter app values one by one

This section follows the practical style of the earlier interactive tester: it asks for one value at a time and shows a default in brackets. Press Enter to use the default. Histories should be entered as comma-separated values in oldest-to-newest order, for example `28,29,27`. This form is only for notebook testing; it does not change the mobile app or save user data.

In [10]:
DEFAULT_PAYLOAD = {
    'date_of_birth': '2000-08-15', 'menarche_age': 13, 'height_cm': 165.1,
    'weight_kg': 60.0, 'sleep_hours': 6.0, 'stress_level': 3,
    'exercise_frequency': 1, 'uses_medication_or_contraceptive': False,
    'cycle_lengths': [28, 29, 27], 'period_lengths': [5, 5, 6],
}

def get_input(prompt, default_value, converter=str):
    entered = input(f'{prompt} [{default_value}]: ').strip()
    if not entered:
        return default_value
    try:
        return converter(entered)
    except ValueError as error:
        raise ValueError(f'Invalid value for {prompt}: {entered}') from error

def get_history(prompt, default_value):
    default_text = ','.join(map(str, default_value))
    text = input(f'{prompt} [{default_text}]: ').strip()
    if not text:
        return default_value
    try:
        return [float(value.strip()) for value in text.split(',') if value.strip()]
    except ValueError as error:
        raise ValueError('Histories must use comma-separated numeric day values.') from error

def get_boolean(prompt, default_value):
    default_text = 'true' if default_value else 'false'
    value = input(f'{prompt} [{default_text}]: ').strip().lower()
    if not value:
        return default_value
    if value not in {'true', 'false'}:
        raise ValueError('Enter true or false.')
    return value == 'true'

def collect_app_payload():
    print('=== NAVYA Model Prediction Tester ===')
    print('Press Enter to use a default value. Cycle and period histories are oldest-to-newest.')
    print('Stress: 1=Very Low, 2=Low, 3=Moderate, 4=High, 5=Very High')
    print('Exercise: 0=Never, 1=1–2 days/week, 2=3+ days/week')
    return {
        'date_of_birth': get_input('Date of birth (YYYY-MM-DD)', DEFAULT_PAYLOAD['date_of_birth']),
        'menarche_age': get_input('Menarche age', DEFAULT_PAYLOAD['menarche_age'], int),
        'height_cm': get_input('Height (cm)', DEFAULT_PAYLOAD['height_cm'], float),
        'weight_kg': get_input('Weight (kg)', DEFAULT_PAYLOAD['weight_kg'], float),
        'sleep_hours': get_input('Sleep hours', DEFAULT_PAYLOAD['sleep_hours'], float),
        'stress_level': get_input('Stress level (1–5)', DEFAULT_PAYLOAD['stress_level'], int),
        'exercise_frequency': get_input('Exercise frequency (0–2)', DEFAULT_PAYLOAD['exercise_frequency'], int),
        'uses_medication_or_contraceptive': get_boolean('Uses medication or contraceptive? (true/false)', DEFAULT_PAYLOAD['uses_medication_or_contraceptive']),
        'cycle_lengths': get_history('Cycle lengths (oldest-to-newest)', DEFAULT_PAYLOAD['cycle_lengths']),
        'period_lengths': get_history('Period lengths (oldest-to-newest)', DEFAULT_PAYLOAD['period_lengths']),
    }

RUN_INTERACTIVE_FORM = False  # Change to True, then run this cell to enter a test user.
if RUN_INTERACTIVE_FORM:
    interactive_payload = collect_app_payload()
    interactive_result = show_prediction(interactive_payload)

## Interpretation

A result labelled `machine_learning` comes from the final exported model and includes its conformal interval. A fallback label is intentional: it avoids presenting a confident ML output when there are too few cycles or the recent history is outside the training support range.